In [30]:
!pip install arxiv

In [31]:
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

In [33]:
import wikipedia

# 1. Set a custom User-Agent to bypass Wikimedia's rate-limiting block
wikipedia.set_user_agent("MyLangChainAgent/1.0 (contact@example.com)")

api_wrapper=WikipediaAPIWrapper(top_k_results=5,doc_content_chars_max=1000)
wiki=WikipediaQueryRun(api_wrapper=api_wrapper)
wiki.run("What is the capital of India?")

"Page: Delhi\nSummary: Delhi, officially the National Capital Territory (NCT) of Delhi, is a megacity and a union territory of India containing New Delhi, the capital of India. Straddling the Yamuna river, but spread chiefly to the west, or beyond its right bank, Delhi shares borders with the state of Uttar Pradesh in the east and with the state of Haryana in the remaining directions. Delhi became a union territory on 1 November 1956 and the NCT in 1995. The NCT covers an area of 1,484 square kilometres (573 sq mi). According to the 2011 census, Delhi's city proper population was over 11 million, while the NCT's population was about 16.8 million. \nThe topography of the medieval fort Purana Qila on the banks of the river Yamuna matches the literary description of the citadel Indraprastha in the Sanskrit epic Mahabharata; however, excavations in the area have revealed no signs of an ancient built environment.\nFrom the early 13th century until the mid-19th century, Delhi was the capital

In [34]:
from langchain_community.document_loaders import WebBaseLoader
loader=WebBaseLoader("https://en.wikipedia.org/wiki/Artificial_intelligence")
from langchain_community.vectorstores import Chroma
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
import os
from dotenv import load_dotenv

load_dotenv()

os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")

gemini_embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")

loader=WebBaseLoader("https://docs.smith.langchain.com/")
docs=loader.load()
texts=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200).split_documents(docs)
db=Chroma.from_documents(texts,gemini_embeddings)
retriever=db.as_retriever()

In [35]:
from langchain_core.tools import create_retriever_tool

retriever_tool = create_retriever_tool(
    retriever,
    "langsmith_search",
    "Search for Smith Documentation Retriever"
)
retriever_tool.name

'langsmith_search'

In [36]:
from langchain_community.utilities import ArxivAPIWrapper
from langchain_community.tools import ArxivQueryRun


import arxiv

# Fix Arxiv Versioning & HTTP 301 Redirect Error
# arXiv recently forced secure HTTPS connections, but the package still tries to use HTTP.
# This forces the package to use the secure URL, bypassing the crash completely.
if hasattr(arxiv, "Client"):
    arxiv.Search.results = lambda self: arxiv.Client(
        query_url_format="https://export.arxiv.org/api/query?{}"
    ).results(self)
else:
    # Fallback just in case you are still on the older arxiv version
    arxiv.root_url = "https://export.arxiv.org/api/"

arxiv_wrapper = ArxivAPIWrapper(top_k_results=5, doc_content_chars_max=1000)
arxiv = ArxivQueryRun(api_wrapper=arxiv_wrapper)

In [37]:
tools=[wiki, arxiv, retriever_tool]
tools

[WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(wiki_client=<module 'wikipedia' from '/opt/anaconda3/envs/genai_env/lib/python3.11/site-packages/wikipedia/__init__.py'>, top_k_results=5, lang='en', load_all_available_meta=False, doc_content_chars_max=1000)),
 ArxivQueryRun(api_wrapper=ArxivAPIWrapper(arxiv_search=<class 'arxiv.arxiv.Search'>, arxiv_exceptions=(<class 'arxiv.arxiv.ArxivError'>, <class 'arxiv.arxiv.UnexpectedEmptyPageError'>, <class 'arxiv.arxiv.HTTPError'>), top_k_results=5, ARXIV_MAX_QUERY_LENGTH=300, continue_on_failure=False, load_max_docs=100, load_all_available_meta=False, doc_content_chars_max=1000)),
 StructuredTool(name='langsmith_search', description='Search for Smith Documentation Retriever', args_schema=<class 'langchain_core.tools.retriever.RetrieverInput'>, func=<function create_retriever_tool.<locals>.func at 0x1480f0900>, coroutine=<function create_retriever_tool.<locals>.afunc at 0x1480f0040>)]

In [38]:
from langchain_google_genai import ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", api_key=os.getenv("GOOGLE_API_KEY"))

In [39]:
# Create a LangSmith API in Settings > API Keys
# Make sure API key env var is set:
# import os; os.environ["LANGSMITH_API_KEY"] = "<your-api-key>"
from langsmith import Client
os.environ["LANGCHAIN_API_KEY"]=os.getenv("LANGCHAIN_API_KEY")

client = Client()
prompt = client.pull_prompt(
    "hwchase17/openai-functions-agent", 
    dangerously_pull_public_prompt=True
)
prompt.messages

[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are a helpful assistant'), additional_kwargs={}),
 MessagesPlaceholder(variable_name='chat_history', optional=True),
 HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={}),
 MessagesPlaceholder(variable_name='agent_scratchpad')]

In [40]:
from langchain_classic.agents import create_tool_calling_agent, AgentExecutor

# 1. Create the Tool Calling Agent
agent = create_tool_calling_agent(llm, tools, prompt)

# 2. Create the Agent Executor (This is the engine that runs the Thought -> Action loop)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

# 3. Give your Agent a complex task!
response = agent_executor.invoke({
    "input": "Tell me about the history of Artificial Intelligence from Wikipedia, and also check Arxiv for any recent papers on it."
})

print(response["output"])



> Entering new AgentExecutor chain...

Invoking: `wikipedia` with `{'query': 'History of Artificial Intelligence'}`




JSONDecodeError: Expecting value: line 1 column 1 (char 0)